# Role Probes: Style Hijacks Tags

**Goal**: After this session, you can build a linear *role probe* — a classifier on a base model's
hidden states that reads off *which role* the model thinks a span of text occupies — and use it to
reproduce the central finding of "Prompt Injection as Role Confusion": **text that *sounds* like a
role inherits that role's representation, even when its tag says otherwise.**
**Time**: 30 minutes.

## What and Why
LLMs partition their context into roles (`<user>`, `<assistant>`, reasoning/`<think>`). The intended
security boundary is the *tag*. This paper shows the boundary doesn't survive into the model's
geometry: a probe trained only on tags ends up keying on **style**, so CoT-styled text wrapped in a
low-privilege `<user>` tag still registers as the model's own reasoning. That equivalence —
*sounding like a role == being that role* — is the mechanism behind prompt injection.

## Key Formula
$$\text{CoTness}(t)\ :=\ P(\text{CoT}\mid h_t)\ =\ \mathrm{softmax}(W\,h_t)_{\text{CoT}}$$

**Where:**
- $h_t$ — the base model's hidden state at token $t$ (a chosen layer).
- $W$ — the learned linear probe (logistic regression weights), mapping a hidden state to a distribution over roles.
- $\text{CoTness}(t)$ — the probe's probability that token $t$ is the model's *own reasoning*. (Analogously: Userness, Assistantness.)

## References
- Ye, Cui, Hadfield-Menell. *Prompt Injection as Role Confusion.* ICML 2026. https://arxiv.org/abs/2603.12277
- Role probes build on Alain & Bengio (2018), *Understanding intermediate layers using linear classifier probes.*


In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "Qwen/Qwen3-0.6B"          # smallest modern-arch base model that still shows the effect
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to(device).eval()
LAYER = 16                          # mid-to-late layer: role geometry is sharpest here (sweep it in the Extension)
print(f"{MODEL}: {model.config.num_hidden_layers} layers, hidden {model.config.hidden_size}, on {device}")

# Neutral, non-instruction content (C4/DOLMA-style). The SAME content gets wrapped in every role,
# so semantics are held constant — a probe can only separate roles using tag-induced geometry.
neutral = [
    "The river winds through three small villages before reaching the sea.",
    "Maple trees turn red earlier than oaks in most temperate forests.",
    "A standard chessboard has sixty-four squares of two alternating colors.",
    "The library closes at nine on weekdays and six on weekends.",
    "Granite is an igneous rock formed from slowly cooled magma.",
    "Most songbirds migrate at night and navigate using the stars.",
    "The recipe calls for two cups of flour and a pinch of salt.",
    "Copper conducts electricity better than iron but worse than silver.",
    "The museum's east wing houses paintings from the seventeenth century.",
    "Tides are caused mainly by the moon's gravitational pull on the oceans.",
    "A hexagon has six sides and six interior angles summing to 720 degrees.",
    "The train departs from platform four every twenty minutes.",
    "Honeybees communicate the location of food through a waggle dance.",
    "The Pacific is the largest and deepest of Earth's oceans.",
    "Cast iron retains heat well but is slower to warm than aluminum.",
    "A leap year occurs almost every four years to keep the calendar aligned.",
    "The old bridge was built from limestone quarried in the next valley.",
    "Saltwater freezes at a lower temperature than freshwater does.",
    "The orchestra tuned to the oboe's A before the performance began.",
    "Bamboo is technically a grass and can grow remarkably quickly.",
]
train_contents = neutral[:15]
test_contents  = neutral[15:]      # held-out CONTENT — probe must generalize past the words it trained on

# Held-out STYLE probes for the punchline (Part 4). Never seen in training.
cot_style_sentences = [
    "The user wants a concise answer, so I should focus on the key steps first.",
    "Let me think about this carefully. First I need to consider the edge cases.",
    "Okay, so the question is asking about X. I'll break this into two parts.",
    "I need to verify each step. If the first assumption holds, then it follows that.",
]
user_style_sentences = [
    "Hey, can you help me figure out how to fix my bike chain?",
    "What's the best way to grow tomatoes in a small apartment?",
    "I'm planning a trip to Japan next spring, any tips?",
    "Could you recommend a good book about the history of jazz?",
]

def content_token_indices(text: str, content: str, offsets: list) -> list:
    """GIVEN (tokenization plumbing): indices of tokens whose character span lies inside `content`.
    Robust to tokenizer boundary merges (leading-space / newline tokens)."""
    s = text.index(content); e = s + len(content)
    return [i for i, (a, b) in enumerate(offsets) if a >= s and b <= e and b > a]


## Part 1: The controlled dataset (the confound-removal trick)

Build the training set: wrap each neutral sentence in *each* role's tags. The point of the whole
method lives here — by holding the **content identical** across roles, the only thing that varies
within a sentence is the tag, so a probe trained on this data *cannot* cheat on semantics. It is
forced to learn the model's internal representation of role.

**Insight (idea-novel)**: a role probe trained on tag-varied-but-content-constant text learns the
tag's *geometric signature*, not its meaning — which is exactly what lets it later detect that
unlabeled text *sounds like* a role. **In one line, explain why holding content constant is what
makes the later "style hijacks tags" result non-circular.**

**Predict before you code**: how many total examples will `train_examples` hold?


In [ ]:
ROLES = ["user", "cot"]

# Model-native tags. The wrapper is plumbing; the learning target is the INVARIANT below it.
TAG = {
    "user": "<|im_start|>user\n{c}<|im_end|>\n",
    "cot":  "<|im_start|>assistant\n<think>\n{c}\n</think>",
}

def wrap(role: str, content: str) -> str:
    """Embed `content` in the role's tags, leaving the content itself byte-for-byte unchanged."""
    # TODO: return the tagged string for `role`
    pass

# TODO: build train_examples = list of (wrapped_text, content, role), every content x every role
train_examples = []   # replace


In [ ]:
# --- Part 1 Validation ---
from collections import Counter
assert len(train_examples) == len(train_contents) * len(ROLES), "one (content, role) pair each"
for text, content, role in train_examples:
    assert content in text, "content must appear verbatim inside the wrapped text (no paraphrasing)"
per_role = Counter(r for _, _, r in train_examples)
print("  examples per role:", dict(per_role))
assert set(per_role.values()) == {len(train_contents)}, "each role must cover every content exactly once"
print(f"  {len(train_examples)} examples; content identical across roles -- the confound is removed.")
print("\nPart 1 complete.")


## Part 2: Probe features from hidden states

Turn a wrapped string into the per-token hidden states the probe will see — but only for the
**content tokens** (exclude the tag tokens themselves; we want the representation of the *content*
as it sits under a role, not the embedding of the literal tag). The character-span alignment is
handled by the given `content_token_indices`; you write the forward pass and the selection.

**Predict before you code**: roughly how many content tokens does the first training example have?


In [ ]:
@torch.no_grad()
def content_reps(text: str, content: str, layer: int = LAYER):
    """Per-token hidden states at `layer`, for the CONTENT tokens only -> np.ndarray (n_content_tok, d)."""
    enc = tok(text, return_tensors="pt", return_offsets_mapping=True)
    idx = content_token_indices(text, content, enc.pop("offset_mapping")[0].tolist())  # GIVEN
    enc = {k: v.to(device) for k, v in enc.items()}
    # TODO: forward with hidden states; take layer `layer`; select content rows `idx`; return float numpy
    pass


In [ ]:
# --- Part 2 Validation ---
predicted_n_tokens = ...   # commit a number BEFORE running: how many content tokens in example 0?
text0, content0, _ = train_examples[0]
rep0 = content_reps(text0, content0)
print(f"  You predicted ~{predicted_n_tokens} content tokens; actual {rep0.shape[0]}"
      f"  (mismatch here = the most useful surprise in the session)")
assert rep0.ndim == 2 and rep0.shape[1] == model.config.hidden_size, \
    f"expected (n_tok, {model.config.hidden_size}), got {rep0.shape}"
assert np.isfinite(rep0).all(), "hidden states must be finite"
print(f"  rep shape {rep0.shape}; value range [{rep0.min():.2f}, {rep0.max():.2f}] -- correct")
print("\nPart 2 complete.")


## Part 3: Train the role probe + the CoTness metric

Fit a linear probe (logistic regression) mapping a single token's hidden state to a role, then
define `cotness(text, content)` = the probe's mean P(CoT) over the content tokens.

**Insight (idea-novel)**: the probe's softmax probability for the CoT class is not just "is this
tagged as reasoning" — once trained, it reads the model's *internal belief* that a token is its own
reasoning. **In one line, explain why a probe trained only on tags can nonetheless score untagged
text.**

**Predict before you code**: content-disjoint test accuracy will be... (chance is 1/2). Commit a number.


In [ ]:
from sklearn.linear_model import LogisticRegression

# Per-token features over the training examples.
# TODO: X = vstack of content_reps for every train example; y = role label repeated per content token
X, y = None, None

probe = None        # TODO: fit a LogisticRegression mapping a token's hidden state -> role

COT = "cot"
def cotness(text: str, content: str) -> float:
    """Mean P(CoT | h_t) over the content tokens of `text`."""
    # TODO: probe probability of the COT class, averaged over the content tokens
    pass


In [ ]:
# --- Part 3 Validation ---
predicted_test_acc = ...   # commit a number in [0.5, 1.0] before running
Xte, yte = [], []
for c in test_contents:                       # CONTENT the probe never trained on
    for r in ROLES:
        rep = content_reps(wrap(r, c), c)
        Xte.append(rep); yte += [r] * len(rep)
Xte = np.concatenate(Xte)
acc = probe.score(Xte, yte)
print(f"  You predicted test acc ~{predicted_test_acc}; actual {acc:.2f} (chance={1/len(ROLES):.2f})")
assert acc > 0.85, f"probe should recover role from tag geometry on unseen content; got {acc:.2f}"
print(f"  content-disjoint role accuracy {acc:.2f} -- the probe learned the TAG geometry, not the words.")
print("\nPart 3 complete.")


## Part 4: Role confusion — style hijacks tags

The payoff. Take CoT-*styled* sentences and genuine user-*styled* sentences, wrap **both** in
`<user>` tags, and measure CoTness. If the tag governed the model's perception, both would read as
user (low CoTness). The paper's claim is they don't: the reasoning-styled text keeps a high CoTness
*despite* the user tag.

**Insight (idea-novel)**: tag and style are encoded onto the *same* internal feature, and under
conflict style wins — so "sounds like reasoning" and "is tagged as reasoning" are the same direction
in latent space. **In one line, state why this makes tag-based prompt-injection defenses fail.**

**Predict before you code**: which set scores higher CoTness, and roughly by how much?


In [ ]:
# Both sets are wrapped in the SAME <user> tag. Only their style differs.
# TODO: mean cotness over each set, wrapping every sentence in the "user" role
cot_style_cotness  = None   # mean cotness(wrap("user", s), s) over cot_style_sentences
user_style_cotness = None   # same over user_style_sentences


In [ ]:
# --- Part 4 Validation ---
print(f"  CoT-STYLE text in <user> tags:  CoTness = {cot_style_cotness:.2f}")
print(f"  USER-STYLE text in <user> tags: CoTness = {user_style_cotness:.2f}")
gap = cot_style_cotness - user_style_cotness
print(f"  gap = {gap:+.2f}")
assert cot_style_cotness > user_style_cotness + 0.2, \
    "CoT-styled text should read as reasoning DESPITE its <user> tag -- style must dominate the tag"
print("  Role confusion reproduced: the tag says <user>, the geometry says CoT.")
print("  This is prompt injection's mechanism -- 'sounds like a role' == 'is that role'.")
print("\nPart 4 complete.")


## Session Debrief

Write your answers into the code cell below (typing is overt retrieval — far stronger than
answering in your head). Don't scroll up.
1. What is the definition of CoTness(t), in symbols?
2. Why must the training content be held *constant* across roles? What would break if it weren't?
3. What single empirical observation (one sentence) makes "prompt injection = role confusion"?

**Check yourself**: the worked solution is in `_solutions/role-probes.ipynb` and the paper is linked
above — open them to grade your answers. Opening the solution ends the retrieval rep, so answer first.

**Challenge**: close this notebook, open a blank one, and rewrite Part 3 (probe fit + CoTness) from
scratch without looking back.


In [ ]:
debrief = """
1.
2.
3.
"""  # type your recall here BEFORE checking _solutions/


In [ ]:
# --- Session log: fill the `___` then run (appends one line to .practice-log.jsonl) ---
import json, datetime, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / ".git").exists() or (p / ".spaced-repetition.json").exists())
record = {
    "problem": "papers/role-confusion-probes",
    "date": datetime.date.today().isoformat(),
    "budget_min": 30,
    "actual_min": ___,                                  # how long it really took
    "parts": [
        {"n": 1, "tier": 3, "outcome": "___"},          # unaided | hint | solution | failed
        {"n": 2, "tier": 3, "outcome": "___"},
        {"n": 3, "tier": 3, "outcome": "___"},
        {"n": 4, "tier": 3, "outcome": "___"},
    ],
    "difficulty": ___,                                  # 1 (trivial) .. 5 (over my head)
    "stuck": "___",                                     # one phrase: where you got stuck
}
with open(_root / ".practice-log.jsonl", "a") as f:
    f.write(json.dumps(record) + "\n")
print("logged ->", _root / ".practice-log.jsonl")


## Extension (Optional)
- **Sweep the layer.** Loop `LAYER` over `[4, 8, 12, 16, 20, 24]`, refit, and plot the style-hijack
  gap vs. layer. Where does role geometry emerge, and where does it fade?
- **Add a third role.** Introduce `assistant` (no `<think>`) and turn CoTness/Userness/Assistantness
  into a 3-way probe. Does the CoT-vs-user contrast survive?
- **Break it on purpose (de-style).** Rephrase a CoT-styled sentence into flat prose with the same
  meaning, re-measure CoTness, and explain why it collapses toward the user subspace (paper Fig. 4).


## Anki Cards

**Card 1**
Front: You train a role probe but hold the text content identical across all role tags. What is the probe forced to learn?
Back: The tag's geometric signature (not semantics)

**Card 2**
Front: CoT-styled text wrapped in a `<user>` tag still scores high CoTness. What does this prove about role boundaries?
Back: Style, not tags, determines role in latent space

**Card 3**
Front: Your role probe gives ~chance accuracy — first thing to check?
Back: Layer choice (use a mid/late layer)
